# Textklassifikation

Im Skript 04 haben wir Strukturen ohne Labels entstehen lassen. Dieses Skript geht den umgekehrten Weg: Ausgehend von Dokumenten, die bereits von Menschen gelabelt wurden, trainieren wir ein Modell, das dieselben Labels auf neue Dokumente anwendet. Das ist **überwachtes Lernen**, und es ist das Arbeitspferd des angewandten Text Mining - Spamfilter, die Weiterleitung von Support-Tickets, die Moderation von Inhalten und die Codierung offener Antworten in Umfragen funktionieren alle auf diese Weise.

Die Aufgabe: **Sage das ORF-eigene Ressort aus einer Schlagzeile vorher.** Die ORF-Redaktion ordnet jede Meldung einem Ressort zu - *Ausland*, *Inland*, *Wirtschaft* usw. -, sodass wir tausende Labels haben, die wir nicht selbst erstellen mussten. Die Skripte 06 und 07 greifen genau diese Aufgabe mit genau dieser Aufteilung der Daten wieder auf, damit sich ihre Methoden unter gleichen Bedingungen vergleichen lassen.

Am Ende dieses Skripts kannst du:

- eine Klassifikationsaufgabe mit einer **Baseline** und einer ehrlichen **Aufteilung in Trainings- und Testdaten** aufsetzen;
- scikit-learn-**Pipelines** vom Text bis zur Vorhersage bauen;
- **Naive Bayes, logistische Regression und lineare SVMs** vergleichen;
- **Zeichen-N-Gramme** nutzen, um mit deutschen Komposita umzugehen;
- mit **Precision, Recall, F1** und einer **Konfusionsmatrix** evaluieren;
- ablesen, was ein lineares Modell gelernt hat, und seine Fehler analysieren.

## Präambel

Wir laden alle Pakete und Module, die wir im Folgenden benötigen.

In [ ]:
# Datentransformation und Visualisierung
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Pakete für Klassifikation und Evaluation
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC

# Hilfsfunktionen des Kurses
import textmining as tm

# Darstellungsoptionen für Tabellen und Grafiken
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Die Aufgabe und die Daten

`tm.subject_task()` liefert ORF-Meldungen mit einer Spalte `label`: die fünf größten Ressorts sowie *Sonstiges*, das den langen Rest der kleinen Ressorts zusammenfasst, die zu wenige Beispiele haben, um daraus zu lernen.

In [ ]:
# Lade die Trainings- und Testdaten (zeitlich aufgeteilt)
train, test = tm.subject_task()
print(f"Trainingsdaten: {len(train):,} Einträge, {train['published'].min():%d %b} - {train['published'].max():%d %b}")
print(f"Testdaten:      {len(test):,} Einträge, {test['published'].min():%d %b} - {test['published'].max():%d %b}")

# Vergleiche die Anteile der Klassen in beiden Teilen
pd.DataFrame({"Training": train["label"].value_counts(normalize=True), "Test": test["label"].value_counts(normalize=True)}).round(2)

Die Klassen sind stark **unausgewogen**: Etwa die Hälfte aller Meldungen ist *Ausland*. Deshalb ist die Accuracy allein irreführend, und deshalb sollte jedes Klassifikationsprojekt mit einer Baseline beginnen.

### Baseline: immer die häufigste Klasse raten

Ein Klassifikator, der den Text ignoriert und immer *Ausland* vorhersagt, legt die Untergrenze fest. Jedes echte Modell muss ihn schlagen, und zwar mit einem Abstand, der den Mehraufwand rechtfertigt.

In [ ]:
# Trainiere einen Klassifikator, der immer die häufigste Klasse vorhersagt
majority = DummyClassifier(strategy="most_frequent").fit(train["title"], train["label"])
pred = majority.predict(test["title"])
print(f"Mehrheits-Baseline: Accuracy {accuracy_score(test['label'], pred):.2f}, "
      f"Makro-F1 {f1_score(test['label'], pred, average='macro'):.2f}")

Zwei Kennzahlen begleiten uns durch das ganze Skript:

- **Accuracy** - der Anteil der richtig gelabelten Meldungen. Die Baseline liegt schon bei etwa der Hälfte, ohne ein einziges Wort zu lesen.
- **Makro-F1** - der F1-Wert (das harmonische Mittel von Precision und Recall), *pro Klasse* berechnet und dann gemittelt, sodass jede Klasse gleich viel zählt, egal wie klein sie ist. Die Baseline erreicht hier fast null, weil sie fünf von sechs Klassen komplett falsch zuordnet. Bei unausgewogenen Daten ist der Makro-F1 die Zahl, auf die wir achten.

## Ein erstes Modell

Eine scikit-learn-**Pipeline** verkettet die Schritte vom Rohtext bis zur Vorhersage, sodass das Ganze als ein Objekt trainiert, evaluiert und wiederverwendet werden kann. Hier: TF-IDF über Wörter und Wortpaare, danach eine logistische Regression.

Für die Tokenisierung verwenden wir diesmal die eingebaute Tokenisierung des Vektorisierers, mit `lowercase=True`. In den Skripten 01-04 haben wir die Tokens von Hand aufbereitet, um jeden Schritt sichtbar zu machen; ein Klassifikator reagiert weniger empfindlich auf diese Entscheidungen, und eine in sich geschlossene Pipeline lässt sich direkt auf neue Schlagzeilen anwenden.

In [ ]:
# Definiere die Wort-Features: TF-IDF über Wörter und Wortpaare
def word_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                           token_pattern=r"(?u)\b[^\W\d_][^\W\d_-]*(?:-[^\W\d_]+)*\b")


# Trainiere die Pipeline und evaluiere sie auf den Testdaten
logreg = make_pipeline(word_features(), LogisticRegression(max_iter=2000, C=10))
logreg.fit(train["title"], train["label"])
pred = logreg.predict(test["title"])
print(f"Logistische Regression: Accuracy {accuracy_score(test['label'], pred):.2f}, "
      f"Makro-F1 {f1_score(test['label'], pred, average='macro'):.2f}")

Das Token-Muster ist der an das Deutsche angepasste reguläre Ausdruck aus Skript 01 (Buchstaben einschließlich Umlauten und ß, Komposita mit Bindestrich bleiben ganz) in der Form, die scikit-learn erwartet. `sublinear_tf` ersetzt die rohen Häufigkeiten durch 1 + log(Häufigkeit) und dämpft so den Einfluss eines Wortes, das sich innerhalb eines Dokuments wiederholt.

## Warum die Aufteilung zeitlich erfolgen muss

Die Aufteilung oben ist **zeitlich**: Das Modell wird auf den älteren drei Vierteln des Archivs trainiert und auf dem jüngsten Viertel getestet - so, wie es auch im Einsatz wäre. Die gängige Alternative ist eine zufällige Aufteilung. Vergleichen wir:

In [ ]:
# Teile alle Daten zufällig auf (gleiche Größe der Testdaten, gleiche Klassenanteile)
everything = pd.concat([train, test], ignore_index=True)
r_train, r_test = train_test_split(everything, test_size=len(test), random_state=0, stratify=everything["label"])

# Trainiere dasselbe Modell auf der zufälligen Aufteilung
random_model = make_pipeline(word_features(), LogisticRegression(max_iter=2000, C=10)).fit(r_train["title"], r_train["label"])
r_pred = random_model.predict(r_test["title"])
print(f"zufällige Aufteilung: Makro-F1 {f1_score(r_test['label'], r_pred, average='macro'):.2f}")
print(f"zeitliche Aufteilung: Makro-F1 {f1_score(test['label'], pred, average='macro'):.2f}")

Eine zufällige Aufteilung sieht meist besser aus, und der Unterschied ist **Data Leakage**, kein Können. Nachrichten wiederholen sich: Eine Geschichte zieht Folgemeldungen, Aktualisierungen und fast identische Schlagzeilen an aufeinanderfolgenden Tagen nach sich. Eine zufällige Aufteilung verstreut die Schlagzeilen jeder Geschichte über Trainings- und Testdaten, sodass das Modell teilweise an Schlagzeilen getestet wird, die es praktisch schon gesehen hat. Eine zeitliche Aufteilung zwingt es, mit Geschichten umzugehen, die erst nach dem Ende seiner Trainingsdaten aufgekommen sind - genau die Situation, der es im echten Einsatz gegenüberstünde. Der Abstand zwischen den beiden Zahlen zeigt, wie stark die zufällige Aufteilung die Leistung überschätzt hätte.

## Klassifikatoren vergleichen

Drei klassische lineare Modelle für Text (logistische Regression und SVM kennen wir bereits aus dem Skript `03-data-mining-classification`):

- **Naive Bayes** zählt, wie oft jedes Wort pro Klasse vorkommt, und nimmt an, dass die Wörter bei gegebener Klasse unabhängig voneinander sind. Grob, schnell und bei kurzen Texten eine starke Baseline. `ComplementNB` ist die Variante, die am besten mit unausgewogenen Klassen zurechtkommt.
- **Logistische Regression** lernt ein Gewicht pro Wort und Klasse und gibt einigermaßen kalibrierte Wahrscheinlichkeiten aus.
- **Lineare SVM** sucht die Grenze mit dem breitesten Abstand (Margin) zwischen den Klassen. Bei dünn besetzten Text-Features oft das genaueste der drei Modelle.

In [ ]:
# Definiere die drei Modelle, jeweils mit denselben Wort-Features
models = {
    "Naive Bayes": make_pipeline(word_features(), ComplementNB(alpha=0.3)),
    "Logistische Regression": make_pipeline(word_features(), LogisticRegression(max_iter=2000, C=10)),
    "Lineare SVM": make_pipeline(word_features(), LinearSVC(C=0.5)),
}

# Trainiere jedes Modell und evaluiere es auf den Testdaten
results = {}
for name, model in models.items():
    p = model.fit(train["title"], train["label"]).predict(test["title"])
    results[name] = {"Accuracy": accuracy_score(test["label"], p), "Makro-F1": f1_score(test["label"], p, average="macro")}
pd.DataFrame(results).T.round(3)

Bei einem Datensatz dieser Größe liegen die drei meist nur wenige Punkte auseinander. Das ist typisch: **Die Repräsentation ist wichtiger als der Klassifikator.** Im nächsten Abschnitt ändern wir die Repräsentation.

## Zeichen-N-Gramme: ein Blick ins Innere deutscher Wörter

Im Skript 01 haben wir gesehen, dass Komposita einen Begriff auf viele seltene Tokens verteilen: *Spritpreis*, *Spritpreisbremse*, *Spritpreisdeckel*. Ein Modell auf Wortebene behandelt diese als voneinander unabhängig, und ein Kompositum, das im Training nicht vorkam, ist für das Modell unsichtbar.

**Zeichen-N-Gramme** beheben einen großen Teil davon ohne jede Zerlegung der Komposita. Mit `analyzer="char_wb"` zerlegt der Vektorisierer jedes Wort in überlappende Fragmente von 2 bis 5 Zeichen. Verwandte Komposita teilen viele Fragmente (`spri`, `prit`, `ritp`, ...), sodass Evidenz zwischen ihnen übertragen wird, und ein unbekanntes Kompositum aktiviert trotzdem vertraute Fragmente.

In [ ]:
# Zeige die ersten Zeichen-N-Gramme eines Kompositums
print(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4)).build_analyzer()("Spritpreisbremse")[:14])


# Definiere die Zeichen-Features: TF-IDF über Fragmente von 2 bis 5 Zeichen
def char_features():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)


# Vergleiche Wörter, Zeichen und die Kombination aus beiden (jeweils mit linearer SVM)
variants = {
    "Wörter": make_pipeline(word_features(), LinearSVC(C=0.5)),
    "Zeichen": make_pipeline(char_features(), LinearSVC(C=0.5)),
    "Wörter + Zeichen": make_pipeline(
        FeatureUnion([("words", word_features()), ("chars", char_features())]), LinearSVC(C=0.5)
    ),
}
results = {}
for name, model in variants.items():
    p = model.fit(train["title"], train["label"]).predict(test["title"])
    results[name] = {"Accuracy": accuracy_score(test["label"], p), "Makro-F1": f1_score(test["label"], p, average="macro")}
pd.DataFrame(results).T.round(3)

Zeichen-Features helfen beim Deutschen in der Regel, und die Kombination ist oft am besten: Wörter liefern saubere, interpretierbare Evidenz, Zeichen liefern die Evidenz, die in Komposita und flektierten Formen verborgen ist. Eine `FeatureUnion` stellt die beiden Feature-Mengen einfach nebeneinander.

## Tuning, ohne die Testdaten anzurühren

Jedes der Modelle oben hat Einstellungen - `C` steuert, wie stark die SVM regularisiert wird. Würden wir sie nach dem Ergebnis auf den Testdaten wählen, würden die Testdaten ins Modell durchsickern. Stattdessen teilt die **Kreuzvalidierung** die *Trainingsdaten* in Folds, trainiert auf allen bis auf einen und bewertet auf dem zurückgehaltenen Fold, reihum. Die Testdaten bleiben bis ganz zum Schluss unberührt.

In [ ]:
# Suche das beste C per Kreuzvalidierung auf den Trainingsdaten
best_pipeline = make_pipeline(
    FeatureUnion([("words", word_features()), ("chars", char_features())]), LinearSVC()
)
search = GridSearchCV(best_pipeline, {"linearsvc__C": [0.05, 0.1, 0.25, 0.5, 1, 2]},
                      scoring="f1_macro", cv=5, n_jobs=-1)
search.fit(train["title"], train["label"])
print("Bestes C:", search.best_params_["linearsvc__C"], f"| kreuzvalidierter Makro-F1 {search.best_score_:.3f}")

# Evaluiere das beste Modell einmal auf den Testdaten
final = search.best_estimator_
pred = final.predict(test["title"])
print(f"Testdaten: Accuracy {accuracy_score(test['label'], pred):.3f}, Makro-F1 {f1_score(test['label'], pred, average='macro'):.3f}")

Die Folds der Kreuzvalidierung sind hier zufällig gebildet, teilen also das Data Leakage aus dem Abschnitt "Warum die Aufteilung zeitlich erfolgen muss", und der kreuzvalidierte Wert fällt im Vergleich zu den zeitlich getrennten Testdaten tendenziell zu optimistisch aus. Für die Wahl zwischen Einstellungen ist das vertretbar - jede Einstellung wird gleichermaßen geschönt -, berichtet wird aber das Ergebnis auf den Testdaten. (`TimeSeriesSplit` liefert zeitlich geordnete Folds, wenn es darauf ankommt.)

## Evaluation pro Klasse

Ein einzelner Wert verbirgt, welche Klassen funktionieren. Der Classification Report und die Konfusionsmatrix zeigen, wo die Fehler liegen.

In [ ]:
# Zeige Precision, Recall und F1 pro Klasse
print(classification_report(test["label"], pred, digits=2, zero_division=0))

In [ ]:
# Zeichne die Konfusionsmatrix, normalisiert pro Zeile
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(test["label"], pred, normalize="true", values_format=".2f",
                                        cmap="Blues", ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title("Konfusionsmatrix (Zeilen: tatsächliches Label, normalisiert)")
plt.show()

Die Zeilen sind die tatsächlichen Labels, die Diagonale ist also der Recall jeder Klasse. Abseits der Diagonale gibt es ein dominantes Muster: **die erste Spalte**. Jede Klasse wandert in großem Umfang nach *Ausland* ab, bei *Chronik* sogar die meisten Meldungen.

Der Grund liegt darin, was die ORF-Ressorts bedeuten. Sie bezeichnen die **Redaktion**, die eine Meldung abgelegt hat, nicht den Ort des Geschehens. Ein Hurrikan in Florida oder ein Brand in Spanien ist *Chronik* (Vorfälle und Unglücke), obwohl er im Ausland passiert; die Bilanz eines globalen Konzerns ist *Wirtschaft*. Das Modell hat dagegen gelernt, dass ausländische Ortsnamen auf *Ausland* hindeuten - was meistens stimmt, da *Ausland* die Hälfte der Daten ausmacht - und wendet das auf jede Geschichte an, die im Ausland spielt. Das zweite Muster, *Wirtschaft*, das nach *Inland* abwandert, ist dieselbe Art von Überschneidung: Wirtschaftspolitik ist zugleich Wirtschaft und Innenpolitik.

*Sonstiges* ist aus einem anderen Grund schwach: Es fasst nicht zusammenhängende kleine Ressorts zusammen (Technologie, Gesundheit, Religion, Lottozahlen) und hat daher kein zusammenhängendes Vokabular, das sich lernen ließe.

## Was hat das Modell gelernt?

Ein lineares Modell besteht aus einem Gewicht pro Feature und Klasse und lässt sich daher direkt ablesen. Wir trainieren das reine Wort-Modell (Zeichenfragmente sind schwer zu lesen) und listen pro Klasse die stärksten positiven Gewichte auf:

In [ ]:
# Trainiere das reine Wort-Modell und hole Vektorisierer und SVM aus der Pipeline
words_model = make_pipeline(word_features(), LinearSVC(C=0.5)).fit(train["title"], train["label"])
vec, svm = words_model.named_steps["tfidfvectorizer"], words_model.named_steps["linearsvc"]
features = vec.get_feature_names_out()

# Zeige pro Klasse die 12 Features mit den größten Gewichten
pd.DataFrame({cls: features[svm.coef_[i].argsort()[::-1][:12]] for i, cls in enumerate(svm.classes_)})

Die Gewichte sind größtenteils sinnvoll - österreichische Parteien und Minister für *Inland*, ausländische Hauptstädte und Staatsspitzen für *Ausland*, Unternehmen und Preise für *Wirtschaft* -, und einige sind aufschlussreich:

- **Deutsche Politik ist Auslandsnachricht.** Der Name des deutschen Bundeskanzlers ist ein starkes Feature für *Ausland*. Von Wien aus gesehen liegt Berlin im Ausland.
- **Formate sickern in die Labels durch.** Suche unter *Inland* nach `studio`: Die Interview-Schlagzeilen "… zu Gast im Studio" aus Skript 03 sind dort abgelegt, also hat das Modell ein Sendungsformat gelernt, kein Thema.
- **Das Modell lernt die Nachrichten des Augenblicks.** Es lernt alles, was die Klassen in *diesem* Trainingszeitraum trennt, einschließlich der Namen von Personen und Unternehmen, die gerade in den Nachrichten waren. Genau deshalb lässt die Leistung mit der Zeit nach: Wenn die Nachrichtenlage weiterzieht, sind die Namen nicht mehr nützlich. Ein solches Modell muss neu trainiert werden, wenn sich seine Welt verändert.

## Fehleranalyse

Der letzte Schritt jeder Evaluation ist das Lesen der Fehler.

In [ ]:
# Wähle die falsch klassifizierten Meldungen aus und zeige eine Stichprobe
errors = test.assign(predicted=pred)[pred != test["label"]]
errors[["title", "label", "predicted"]].sample(12, random_state=1)

Die meisten davon werden als *Ausland* vorhergesagt, wie es die Konfusionsmatrix angekündigt hat, und sie lassen sich drei Arten zuordnen:

1. **Redaktionelle Konventionen, die der Text nicht verrät.** Ein Sturm in Florida ist für den ORF *Chronik* und für das Modell *Ausland*. Die Schlagzeile allein kann nicht zeigen, welche Redaktion die Meldung abgelegt hat, und diese Fehler setzen jeder Methode, die nur die Schlagzeile liest, eine Obergrenze für die erreichbare Accuracy.
2. **Fehlender Kontext.** Eine Schlagzeile wie *"Der 7. Oktober und die Schuldumkehr"* ist ein Fragment, dessen Ressort vom Artikel dahinter abhängt.
3. **Echte Modellfehler**, oft bei neuen Namen oder Themen, die im Training nicht vorkamen.

Zu zählen, wie viele Fehler in einer Stichprobe von fünfzig auf jede Gruppe entfallen, ist eine gute Übung, und es sagt mehr über die tatsächliche Qualität des Modells aus als eine weitere Nachkommastelle beim F1.

## Unausgewogene Klassen

Standardmäßig zählt jedes Trainingsbeispiel gleich viel, sodass das Modell zu den großen Klassen tendiert - der Sog in Richtung *Ausland*, den wir oben gesehen haben. `class_weight="balanced"` gewichtet die Beispiele umgekehrt proportional zur Häufigkeit ihrer Klasse. Das tauscht oft etwas Accuracy gegen einen besseren Recall bei kleinen Klassen, wobei der Effekt schwankt und klein sein kann:

In [ ]:
# Trainiere dieselbe Pipeline mit ausgeglichenen Klassengewichten
balanced = make_pipeline(
    FeatureUnion([("words", word_features()), ("chars", char_features())]),
    LinearSVC(C=search.best_params_["linearsvc__C"], class_weight="balanced"),
).fit(train["title"], train["label"])
bpred = balanced.predict(test["title"])

# Vergleiche F1 pro Klasse, Accuracy und Makro-F1 beider Varianten
pd.DataFrame(
    {
        "Standard": f1_score(test["label"], pred, average=None, labels=final.classes_),
        "ausgeglichen": f1_score(test["label"], bpred, average=None, labels=final.classes_),
    },
    index=final.classes_,
).round(2).T.assign(**{
    "Accuracy": [accuracy_score(test["label"], pred), accuracy_score(test["label"], bpred)],
    "Makro-F1": [f1_score(test["label"], pred, average="macro"), f1_score(test["label"], bpred, average="macro")],
}).round(3)

Welche Variante besser ist, hängt davon ab, was die Fehler kosten. Geht es darum, jede *Kultur*-Meldung zu finden, ist das ausgeglichene Modell das richtige, auch bei geringerer Accuracy. Lege fest, worauf es ankommt, bevor du die Kennzahl wählst, nicht danach.

## Zusammenfassung

| Schritt | Lehre |
|---|---|
| zuerst eine Baseline | Raten der häufigsten Klasse legt die Untergrenze fest; die Accuracy allein verschleiert das |
| zeitlich aufteilen | zufällige Aufteilungen lassen wiederholte Geschichten in die Trainingsdaten durchsickern |
| Repräsentation vor Klassifikator | Zeichen-N-Gramme sind wichtiger als NB vs. SVM |
| auf Trainings-Folds tunen | die Testdaten werden einmal verwendet, ganz am Ende |
| Konfusionsmatrix und Fehler lesen | die Labels bezeichnen Redaktionen, nicht Orte - der Text kann nicht immer zeigen, welche |

Das beste Modell aus dem Abschnitt "Tuning, ohne die Testdaten anzurühren" ist der Maßstab für die nächsten beiden Skripte: Skript 06 ersetzt die TF-IDF-Features durch Embeddings, und Skript 07 lässt ein großes Sprachmodell (LLM) dieselbe Aufgabe ganz ohne Training erledigen.

## Übungen

1. **Lernkurve.** Trainiere die finale Pipeline auf 10 %, 25 %, 50 % und 100 % der Trainingsdaten (immer die ältesten Einträge). Wie schnell steigt der Makro-F1, und würden mehr gelabelte Daten helfen?
2. **Beschreibungen.** Etwa ein Fünftel der ORF-Meldungen hat eine Beschreibung. Verbessert sich das Ergebnis bei diesen Meldungen, wenn du `text` statt `title` verwendest?
3. **Nachlassen über die Zeit.** Teile die Testdaten in ihre erste und zweite Hälfte. Ist das Modell auf der späteren Hälfte, die weiter von seinen Trainingsdaten entfernt ist, schlechter?
4. **Ein anderes Ziel.** Trainiere einen Klassifikator, der das *Medium* einer Schlagzeile aus allen vier deutschsprachigen Feeds vorhersagt. Welche Medien lassen sich leicht unterscheiden, und was verraten die wichtigsten Features über den Stil des jeweiligen Hauses?

## Quellen

* https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
* https://scikit-learn.org/stable/modules/naive_bayes.html
* https://scikit-learn.org/stable/modules/svm.html
* https://scikit-learn.org/stable/modules/model_evaluation.html
* https://de.wikipedia.org/wiki/Tf-idf-Ma%C3%9F
* https://pandas.pydata.org/docs/
* https://matplotlib.org/